<a href="https://colab.research.google.com/github/Karsuman4298/Generative-AI/blob/main/Text_Summarizer_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Mar 21 09:48:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate

Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


# accelerate helps uses cuda gpu

In [4]:
!pip install transformers accelerate

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)


In [5]:
!pip install --upgrade ipywidgets nbformat nbconvert
!jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [81]:
from google.colab import output
output.enable_custom_widget_manager()

In [7]:
from transformers import pipeline, set_seed
from datasets import load_dataset,load_from_disk
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer,AutoModelForSequenceClassification
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
model_ckpt="google/pegasus-cnn_dailymail"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
from transformers import AutoModelForSeq2SeqLM

In [11]:
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [12]:
dataset=load_dataset("knkarthick/samsum")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [20]:
dataset['train']['dialogue'][1]

'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great'

In [21]:
dataset['train'][1]['summary']

'Olivia and Olivier are voting for liberals in this election. '

In [27]:
def convert_example_to_token(example_batch):

  input_encodings=tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)
  target_encodings=tokenizer(example_batch['summary'],max_length=128,truncation=True)


  return{
      "input_ids":input_encodings["input_ids"],
      "attention_mask":input_encodings["attention_mask"],
      "labels":target_encodings["input_ids"]
  }

In [28]:
dataset_tokenized=dataset.map(convert_example_to_token,batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [33]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [35]:
dataset_tokenized['train']['input_ids'][1]

[17935,
 48,
 2529,
 24,
 16,
 6125,
 15,
 12,
 33,
 2871,
 49,
 10360,
 48,
 35781,
 27,
 226,
 4,
 17935,
 48,
 2484,
 211,
 1139,
 10360,
 48,
 1406,
 1]

In [36]:
dataset_tokenized['train']['attention_mask'][1]

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [37]:
dataset_tokenized['train']['labels'][1]

[17935, 8, 34193, 24, 6125, 15, 33092, 12, 33, 2871, 4, 7, 1]

In [38]:
dataset_tokenized['train']['summary'][1]

'Olivia and Olivier are voting for liberals in this election. '

In [40]:
from transformers import  DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

In [44]:
seq2seq_data_collator

DataCollatorForSeq2Seq(tokenizer=PegasusTokenizer(name_or_path='google/pegasus-cnn_dailymail', vocab_size=96000, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask_2>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<n>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	96000: AddedToken("<mask_2>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), model=PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (emb

In [46]:
from transformers import TrainingArguments,Trainer

trainer_args=TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,

)

#Using Test data for training because test data is huge

In [48]:
trainer=Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_tokenized['test'],
    eval_dataset=dataset_tokenized['validation']

)

In [49]:
trainer.train()

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=183.34718381441556, metrics={'train_runtime': 623.198, 'train_samples_per_second': 1.314, 'train_steps_per_second': 0.083, 'total_flos': 314203859361792.0, 'train_loss': 183.34718381441556, 'epoch': 1.0})

In [60]:
def generate_batch_sized_chunks(list_of_elements,batch_size):
  for i in range(0,len(list_of_elements),batch_size):
    yield list_of_elements[i:i+batch_size]

def calculate_metric_on_test_ds(dataset,metric,model,tokenizer,batch_size=16,device=device,
                                column_text="article",column_summary="highlights"):
  article_batches=list(generate_batch_sized_chunks(dataset[column_text],batch_size))
  target_batches=list(generate_batch_sized_chunks(dataset[column_summary],batch_size))
  for article_batch,target_batch in tqdm(zip(article_batches,target_batches),total=len(article_batches)):
    inputs=tokenizer(article_batch,max_length=1024,truncation=True,padding="max_length",return_tensors="pt")
    summaries=model.generate(input_ids=inputs["input_ids"].to(device),attention_mask=inputs["attention_mask"].to(device),length_penalty=0.8,num_beams=8,max_length=128)
    decoded_summaries=[tokenizer.decode(s,skip_special_tokens=True,clean_up_tokenization_spaces=True) for s in summaries]
    decoded_summaries=[d.replace(""," ") for d in decoded_summaries]



    metric.add_batch(predictions=decoded_summaries,references=target_batch)
    score=metric.compute()
  return score

In [56]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


# Rouge Score is the Evaluatipn metric for text summarizzation task

In [57]:
import evaluate
rouge_names=["rouge1","rouge2",
             "rougeL","rougeLsum"]

rouge_metric=evaluate.load('rouge')

In [62]:
score=calculate_metric_on_test_ds(dataset['test'][0:10],rouge_metric,trainer.model,tokenizer,batch_size=2, column_text="dialogue", column_summary="summary")

rouge_dict=dict((rn,score[rn]) for rn in rouge_names)

pd.DataFrame(rouge_dict,index=[f'pegasus'])

100%|██████████| 5/5 [00:51<00:00, 10.32s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.012055,0.0,0.012055,0.012055


#Saving the trained model

In [63]:
model_pegasus.save_pretrained('pegasus-samsum-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#Saving the Tokenizer


In [65]:

tokenizer.save_pretrained('tokenizer')

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

#Load


In [78]:
my_tokenizer=AutoTokenizer.from_pretrained("/content/tokenizer")
model=AutoModelForSeq2SeqLM.from_pretrained("/content/pegasus-samsum-model").to(device)

Loading weights:   0%|          | 0/682 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


#Prediction

In [80]:
gen_kwargs={"length_penalty":0.8,"num_beams":8,"max_length":128}
sample_text=dataset['test'][0]['dialogue']

reference=dataset['test'][0]['summary']

# Tokenize the input text
inputs = my_tokenizer(sample_text, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")

# Move inputs to the device
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

# Generate summary
summaries = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs)

# Decode the summary
decoded_summary = my_tokenizer.decode(summaries[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
decoded_summary = decoded_summary.replace("","")

print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(decoded_summary)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
group waye greater run In That quality her New need Co home years area through My through director found first find last way madee greater run In That quality her New need Co home yearsa us years program us years believe us years believe us years program us years program us years program us years program us years program us years program us years program us years program us years program us years program us years program us years program us years program us 